# Gradient descent: from local slope to a linear neuron

**Lecture 10 · Notebook 00 · CMOR 438 / INDE 577**  
**Core:** 155 minutes · **Practice:** 50 minutes · **Extension:** 40+ minutes

The 2022 version of this lesson walked downhill one visible step at a time. We retain that useful pace and make the mathematics explicit: local linear approximation, descent direction, step size, multivariable gradient, loss surface, and finally the training of one identity-activation neuron.

Gradient descent is an **optimization algorithm**, not a model. Linear regression is the model; mean squared error is the objective; derivatives supply local sensitivity; gradient descent changes the parameters.

## How to use this notebook

For every update, name four objects:

1. the parameter currently being changed;
2. the scalar objective being minimized;
3. the derivative or gradient evaluated at the current parameter; and
4. the learning rate that turns direction into a step.

Derive the displayed gradients on paper. Before each plot, predict the direction of the next step. Run all cells in the **Rice DSM** kernel.

**Prerequisites:** derivatives, partial derivatives, vectors, matrix multiplication, and the preceding linear-regression notebook.

## Learning objectives

You will be able to:

- interpret a derivative as the coefficient of a local linear approximation;
- explain why the negative derivative or gradient is a descent direction;
- implement and verify gradient descent for scalar and vector parameters;
- derive a stability interval for a quadratic learning problem;
- visualize optimization in parameter space and model behavior in data space;
- derive the weight and bias gradients of mean squared error;
- prove that an identity-activation neuron trained with squared loss is linear regression;
- diagnose poor scaling, divergence, vanishing progress, and data leakage; and
- distinguish batch, stochastic, and mini-batch gradient descent from automatic differentiation.

## Why this matters in industry

Modern frameworks calculate gradients automatically, but they cannot decide whether the objective represents the scientific goal, whether the data split is valid, whether input scale makes the optimization unstable, or whether a low training loss supports deployment.

Optimization telemetry—loss, gradient norm, parameter norm, learning rate, NaNs, and validation behavior—is often the first evidence available when training fails. Mathematical understanding turns those signals into diagnoses instead of rituals.

## Historical context: optimization predates machine learning

Gradient descent is not a neural-network invention. In 1847 Augustin-Louis Cauchy described a steepest-descent method for solving systems arising from quadratic problems. In 1951 Herbert Robbins and Sutton Monro introduced stochastic approximation for finding a solution when only noisy experimental responses were available. Modern stochastic gradient methods combine these ideas with vectorized computation and automatic differentiation.

This history clarifies the layers:

- calculus defines local sensitivity;
- optimization defines an update procedure;
- statistics explains what noisy data say about a population;
- a model defines the candidate input-output behavior; and
- software makes the procedure reproducible and observable.

Calling every update “learning” can hide which layer actually failed.

In [ ]:
from __future__ import annotations

import sqlite3
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, FancyBboxPatch
from rice_dsm.paths import course_database_path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 438577
rng = np.random.default_rng(RANDOM_SEED)

## 1. A function of one variable

Consider

$$f(\theta)=(\theta-3)^2+1, \qquad f'(\theta)=2(\theta-3).$$

The derivative is more than “slope.” It is the best local linear description:

$$f(\theta+\Delta)\approx f(\theta)+f'(\theta)\Delta$$

for small $\Delta$. If $f'(\theta)>0$, a small negative $\Delta$ lowers the approximation. If $f'(\theta)<0$, a small positive $\Delta$ does. This motivates stepping opposite the derivative:

$$\theta_{t+1}=\theta_t-\eta f'(\theta_t),$$

where $\eta>0$ is the **learning rate** or step size.

In [ ]:
def scalar_objective(parameter: float | np.ndarray) -> float | np.ndarray:
    """Evaluate the one-dimensional quadratic objective."""

    return (parameter - 3.0) ** 2 + 1.0


def scalar_gradient(parameter: float | np.ndarray) -> float | np.ndarray:
    """Evaluate the analytic derivative of ``scalar_objective``."""

    return 2.0 * (parameter - 3.0)


theta_0 = 5.0
finite_difference = (
    scalar_objective(theta_0 + 1e-6) - scalar_objective(theta_0 - 1e-6)
) / (2e-6)
assert np.isclose(finite_difference, scalar_gradient(theta_0), rtol=1e-8)
assert scalar_objective(3.0) == 1.0

In [ ]:
theta_grid = np.linspace(-1, 6.5, 300)
tangent_grid = np.linspace(theta_0 - 1.3, theta_0 + 0.7, 80)
tangent_values = scalar_objective(theta_0) + scalar_gradient(theta_0) * (tangent_grid - theta_0)
theta_1 = theta_0 - 0.15 * scalar_gradient(theta_0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(theta_grid, scalar_objective(theta_grid), label="$f(\\theta)$")
axes[0].plot(tangent_grid, tangent_values, linestyle="--", color="crimson", label="tangent")
axes[0].scatter([theta_0], [scalar_objective(theta_0)], color="black", zorder=3)
axes[0].set(xlabel="$\\theta$", ylabel="$f(\\theta)$", title="Derivative = local slope")
axes[0].legend()

axes[1].plot(theta_grid, scalar_objective(theta_grid))
axes[1].scatter(
    [theta_0, theta_1], [scalar_objective(theta_0), scalar_objective(theta_1)],
    color=["black", "crimson"], zorder=3,
)
axes[1].annotate("negative-gradient step", xy=(theta_1, scalar_objective(theta_1)),
                 xytext=(theta_0, scalar_objective(theta_0)),
                 arrowprops={"arrowstyle": "->", "color": "crimson", "lw": 2})
axes[1].set(xlabel="$\\theta$", ylabel="$f(\\theta)$", title="One update moves downhill")
plt.tight_layout()
plt.show()

### Work the first updates before writing an algorithm

With $\theta_0=5$ and $\eta=0.15$,

$$
\theta_1=5-0.15(4)=4.4,
\qquad
\theta_2=4.4-0.15(2.8)=3.98.
$$

The derivative shrinks as the parameter approaches 3, so fixed learning rate does **not** imply fixed distance traveled. The table below makes the state transition explicit.

In [ ]:
manual_rows = []
theta = theta_0
for step in range(5):
    gradient = float(scalar_gradient(theta))
    next_theta = theta - 0.15 * gradient
    manual_rows.append((step, theta, float(scalar_objective(theta)), gradient, next_theta))
    theta = next_theta

import pandas as pd

manual_trace = pd.DataFrame(
    manual_rows, columns=["step", "theta_t", "f(theta_t)", "gradient", "theta_(t+1)"]
)
assert np.all(np.diff(manual_trace["f(theta_t)"]) < 0)
manual_trace

In [ ]:
def gradient_descent_1d(
    start: float, *, learning_rate: float, steps: int
) -> np.ndarray:
    """Run fixed-step gradient descent on ``scalar_objective``.

    Parameters
    ----------
    start
        Initial scalar parameter.
    learning_rate
        Positive multiplier on the negative derivative.
    steps
        Number of updates.

    Returns
    -------
    numpy.ndarray
        Parameter history, including the initial value.

    Raises
    ------
    ValueError
        If the learning rate or number of steps is not positive.
    """

    if learning_rate <= 0:
        raise ValueError("learning_rate must be positive")
    if steps <= 0:
        raise ValueError("steps must be positive")
    history = [float(start)]
    for _ in range(steps):
        history.append(history[-1] - learning_rate * scalar_gradient(history[-1]))
    return np.asarray(history)


scalar_history = gradient_descent_1d(5.0, learning_rate=0.15, steps=18)
assert scalar_objective(scalar_history[-1]) < scalar_objective(scalar_history[0])
assert np.isclose(scalar_history[-1], 3.0, atol=0.01)

### The learning rate controls stability

Let $e_t=\theta_t-3$. For this quadratic,

$$e_{t+1}=(1-2\eta)e_t.$$

Therefore convergence requires $|1-2\eta|<1$, or

$$0<\eta<1.$$

- $0<\eta<0.5$: approach from one side.
- $0.5<\eta<1$: oscillate with shrinking magnitude.
- $\eta=1$: oscillate forever with constant magnitude.
- $\eta>1$: oscillate and diverge.

This interval is specific to this curvature. There is no universally safe learning rate.

### The curvature rule generalizes

For

$$
J(\theta)=\frac{a}{2}(\theta-\theta^*)^2,\qquad a>0,
$$

the error follows

$$
e_{t+1}=(1-\eta a)e_t.
$$

Therefore a fixed rate converges exactly when $0<\eta<2/a$. Curvature $a$ sets the scale of a safe step.

For a positive-definite quadratic in several variables, diagonalizing its Hessian shows that each eigenvector direction has recurrence

$$
e_{j,t+1}=(1-\eta\lambda_j)e_{j,t}.
$$

One learning rate must satisfy the steepest direction, $0<\eta<2/\lambda_{\max}$. The condition number $\kappa=\lambda_{\max}/\lambda_{\min}$ measures how much slower the flattest direction can be. This is the mathematical source of the zigzagging seen in elongated contours.

In [ ]:
learning_rates = (0.03, 0.35, 0.90, 1.05)
learning_rate_histories = {
    rate: gradient_descent_1d(5.0, learning_rate=rate, steps=18)
    for rate in learning_rates
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, rate in zip(axes.flat, learning_rates, strict=True):
    history = learning_rate_histories[rate]
    ax.plot(theta_grid, scalar_objective(theta_grid), color="0.65")
    ax.plot(history, scalar_objective(history), marker="o", color="crimson", markersize=3)
    ax.set(xlabel="$\\theta$", ylabel="$f(\\theta)$", title=f"learning rate $\\eta={rate}$")
    ax.set_ylim(0, min(35, max(12, float(np.nanmax(scalar_objective(history))))))
plt.tight_layout()
plt.show()

assert scalar_objective(learning_rate_histories[0.35][-1]) < scalar_objective(learning_rate_histories[0.03][-1])
assert scalar_objective(learning_rate_histories[1.05][-1]) > scalar_objective(5.0)

## 2. Functions of multiple variables

For $\boldsymbol\theta=(\theta_1,\theta_2)^T$, partial derivatives measure sensitivity while holding other coordinates fixed. The gradient collects them:

$$
J(\theta_1,\theta_2)=2(\theta_1-2)^2+\tfrac12(\theta_2+1)^2,
$$

$$
\nabla J(\boldsymbol\theta)=
\begin{bmatrix}
4(\theta_1-2)\\
\theta_2+1
\end{bmatrix}.
$$

The directional derivative along a unit vector $\boldsymbol u$ is $\nabla J^T\boldsymbol u$. By Cauchy–Schwarz, its smallest possible value is $-\|\nabla J\|_2$, attained by $\boldsymbol u=-\nabla J/\|\nabla J\|_2$. Thus the negative gradient is the direction of steepest local descent **under Euclidean distance**.

The vector update is

$$\boldsymbol\theta_{t+1}=\boldsymbol\theta_t-\eta\nabla J(\boldsymbol\theta_t).$$

In [ ]:
def vector_objective(parameters: np.ndarray) -> float:
    """Evaluate the anisotropic two-parameter quadratic."""

    first, second = np.asarray(parameters, dtype=float)
    return float(2.0 * (first - 2.0) ** 2 + 0.5 * (second + 1.0) ** 2)


def vector_gradient(parameters: np.ndarray) -> np.ndarray:
    """Return the gradient of ``vector_objective``."""

    first, second = np.asarray(parameters, dtype=float)
    return np.array([4.0 * (first - 2.0), second + 1.0])


vector_history = [np.array([-2.0, 3.5])]
for _ in range(32):
    vector_history.append(vector_history[-1] - 0.18 * vector_gradient(vector_history[-1]))
vector_history = np.asarray(vector_history)

test_point = np.array([0.5, -0.25])
finite_difference_gradient = np.array([
    (vector_objective(test_point + 1e-6 * basis) - vector_objective(test_point - 1e-6 * basis)) / 2e-6
    for basis in np.eye(2)
])
assert np.allclose(finite_difference_gradient, vector_gradient(test_point), rtol=1e-7)
assert np.allclose(vector_history[-1], [2.0, -1.0], atol=0.02)

In [ ]:
first_grid = np.linspace(-3, 4, 170)
second_grid = np.linspace(-3.5, 4.5, 170)
T1, T2 = np.meshgrid(first_grid, second_grid)
J_grid = 2.0 * (T1 - 2.0) ** 2 + 0.5 * (T2 + 1.0) ** 2

fig = plt.figure(figsize=(13, 5))
ax_surface = fig.add_subplot(1, 2, 1, projection="3d")
ax_surface.plot_surface(T1, T2, J_grid, cmap="viridis", alpha=0.75, linewidth=0)
ax_surface.plot(
    vector_history[:, 0], vector_history[:, 1],
    [vector_objective(point) for point in vector_history], color="crimson", marker="o", markersize=3,
)
ax_surface.set(xlabel="$\\theta_1$", ylabel="$\\theta_2$", zlabel="$J$", title="Surface and optimization path")

ax_contour = fig.add_subplot(1, 2, 2)
ax_contour.contour(T1, T2, J_grid, levels=22)
ax_contour.plot(vector_history[:, 0], vector_history[:, 1], color="crimson", marker="o", markersize=3)
initial_gradient = vector_gradient(vector_history[0])
ax_contour.quiver(
    *vector_history[0], *(-initial_gradient), color="black", angles="xy", scale_units="xy", scale=8,
    label="negative gradient",
)
ax_contour.scatter([2], [-1], marker="*", s=150, color="gold", edgecolor="black", label="minimum")
ax_contour.set(xlabel="$\\theta_1$", ylabel="$\\theta_2$", title="Contours reveal unequal curvature")
ax_contour.legend()
plt.tight_layout()
plt.show()

The elongated contours show unequal curvature. One learning rate must be small enough for the steep direction, which can make travel along the shallow direction slow. This is the optimization reason scaling and conditioning matter.

## 3. A single neuron becomes linear regression

A single neuron computes a weighted sum and then an activation:

$$
z=\boldsymbol w^T\boldsymbol x+b,
\qquad
\widehat y=\phi(z).
$$

Choose the **identity activation** $\phi(z)=z$. Then

$$\widehat y=\boldsymbol w^T\boldsymbol x+b,$$

which is exactly the affine model from linear regression. The word “neuron” changes neither the function nor its mathematics. A single identity neuron has no hidden layer and is not yet a neural network.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis("off")
for index, y_position in enumerate((1.1, 2.5, 3.9), start=1):
    node = Circle((1.2, y_position), 0.35, facecolor="#dbeafe", edgecolor="#1d4ed8")
    ax.add_patch(node)
    ax.text(1.2, y_position, f"$x_{index}$", ha="center", va="center")
    ax.annotate("", xy=(4.25, 2.5), xytext=(1.58, y_position), arrowprops={"arrowstyle": "->"})
    ax.text(2.7, (y_position + 2.5) / 2 + 0.15, f"$w_{index}$")
sum_box = FancyBboxPatch((4.25, 1.85), 1.25, 1.3, boxstyle="round,pad=0.08", facecolor="#fef3c7")
activation_box = FancyBboxPatch((6.35, 1.85), 1.5, 1.3, boxstyle="round,pad=0.08", facecolor="#dcfce7")
ax.add_patch(sum_box)
ax.add_patch(activation_box)
ax.text(4.875, 2.5, "$z=\\mathbf{w}^T\\mathbf{x}+b$", ha="center", va="center")
ax.text(7.1, 2.5, "$\\phi(z)=z$", ha="center", va="center")
ax.annotate("", xy=(6.35, 2.5), xytext=(5.5, 2.5), arrowprops={"arrowstyle": "->"})
ax.annotate("", xy=(9.2, 2.5), xytext=(7.85, 2.5), arrowprops={"arrowstyle": "->"})
ax.text(9.4, 2.5, "$\\widehat y$", ha="left", va="center", fontsize=14)
ax.set_title("One identity-activation neuron")
plt.show()

## Worked example: derive the learning rule

We return to the real diabetes dataset from the preceding notebook and use baseline body mass index to predict the quantitative one-year disease-progression score. The one-feature model is intentionally incomplete: its purpose is to make every optimization coordinate visible, not to suggest a clinical system. This time a narrow SQL query retrieves only the columns the experiment requires.

For training examples $(\boldsymbol x_i,y_i)$, choose mean squared error

$$
L(\boldsymbol w,b)=\frac1n\sum_{i=1}^n(\widehat y_i-y_i)^2,
\qquad
\widehat y_i=\boldsymbol w^T\boldsymbol x_i+b.
$$

Let $r_i=\widehat y_i-y_i$. Applying the chain rule gives

$$
\frac{\partial L}{\partial w_j}=\frac{2}{n}\sum_i r_i x_{ij},
\qquad
\frac{\partial L}{\partial b}=\frac{2}{n}\sum_i r_i.
$$

In matrix form,

$$
\nabla_{\boldsymbol w}L=\frac{2}{n}X^T(X\boldsymbol w+b\mathbf1-\boldsymbol y),
\qquad
\frac{\partial L}{\partial b}=\frac{2}{n}\mathbf1^T(X\boldsymbol w+b\mathbf1-\boldsymbol y).
$$

Notice the three stages in each derivative: prediction → residual → sensitivity of prediction to a parameter.

The database path is resolved through the installed course package rather than assumed relative to the kernel's current working directory. This makes the same cell work when VS Code launches the kernel from the repository root, this lecture directory, or another directory. Converting the absolute path with `Path.as_uri()` also avoids operating-system-specific path separators in the SQLite connection URI.

In [ ]:
database_path = course_database_path()
database_uri = f"{database_path.as_uri()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    diabetes_for_optimization = pd.read_sql_query(
        """
        SELECT bmi, disease_progression
        FROM diabetes_observations
        WHERE bmi IS NOT NULL AND disease_progression IS NOT NULL
        ORDER BY observation_id
        """,
        connection,
    )
body_mass_index = diabetes_for_optimization[["bmi"]].to_numpy()
disease_progression = diabetes_for_optimization["disease_progression"].to_numpy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    body_mass_index, disease_progression,
    test_size=0.25, random_state=RANDOM_SEED,
)

scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

assert X_train.shape == (331, 1)
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(X_train.std(axis=0), 1.0, atol=1e-12)

In [ ]:
def linear_neuron_gradients(
    features: np.ndarray, targets: np.ndarray, weights: np.ndarray, bias: float
) -> tuple[np.ndarray, float]:
    """Return full-batch MSE gradients for a linear neuron."""

    residuals = features @ weights + bias - targets
    weight_gradient = (2.0 / len(features)) * features.T @ residuals
    bias_gradient = 2.0 * float(np.mean(residuals))
    return weight_gradient, bias_gradient


trial_weights = np.array([0.4])
trial_bias = 1.5
weight_gradient, bias_gradient = linear_neuron_gradients(
    X_train, y_train, trial_weights, trial_bias
)
epsilon = 1e-6
loss_at = lambda w, b: float(np.mean((X_train @ np.asarray(w) + b - y_train) ** 2))
finite_weight_gradient = (
    loss_at(trial_weights + epsilon, trial_bias) - loss_at(trial_weights - epsilon, trial_bias)
) / (2 * epsilon)
finite_bias_gradient = (
    loss_at(trial_weights, trial_bias + epsilon) - loss_at(trial_weights, trial_bias - epsilon)
) / (2 * epsilon)
assert np.allclose(weight_gradient, finite_weight_gradient, rtol=1e-7)
assert np.isclose(bias_gradient, finite_bias_gradient, rtol=1e-7)

In [ ]:
@dataclass(frozen=True)
class LinearNeuronFit:
    """Parameters and optimization history from a fitted linear neuron."""

    weights: np.ndarray
    bias: float
    loss_history: np.ndarray
    parameter_history: np.ndarray


def fit_linear_neuron(
    features: np.ndarray,
    targets: np.ndarray,
    *,
    learning_rate: float = 0.08,
    epochs: int = 160,
) -> LinearNeuronFit:
    """Fit an identity-activation neuron with batch gradient descent.

    Parameters
    ----------
    features
        Two-dimensional design matrix with observations in rows.
    targets
        One-dimensional continuous targets.
    learning_rate
        Positive fixed step size.
    epochs
        Positive number of full-data updates.

    Returns
    -------
    LinearNeuronFit
        Final parameters and histories including the initial state.

    Raises
    ------
    ValueError
        If shapes, values, or optimization settings are invalid.
    """

    X = np.asarray(features, dtype=float)
    y = np.asarray(targets, dtype=float)
    if X.ndim != 2 or y.ndim != 1 or len(X) != len(y):
        raise ValueError("features must be 2D and align with 1D targets")
    if len(X) == 0 or not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("training data must be nonempty and finite")
    if learning_rate <= 0 or epochs <= 0:
        raise ValueError("learning_rate and epochs must be positive")

    weights = np.zeros(X.shape[1])
    bias = 0.0
    loss_history = [float(np.mean((X @ weights + bias - y) ** 2))]
    parameter_history = [np.concatenate([[bias], weights.copy()])]
    for _ in range(epochs):
        weight_gradient, bias_gradient = linear_neuron_gradients(X, y, weights, bias)
        weights -= learning_rate * weight_gradient
        bias -= learning_rate * bias_gradient
        loss_history.append(float(np.mean((X @ weights + bias - y) ** 2)))
        parameter_history.append(np.concatenate([[bias], weights.copy()]))
    return LinearNeuronFit(
        weights=weights.copy(),
        bias=float(bias),
        loss_history=np.asarray(loss_history),
        parameter_history=np.asarray(parameter_history),
    )


neuron_fit = fit_linear_neuron(X_train, y_train)
assert neuron_fit.loss_history[-1] < neuron_fit.loss_history[0]
assert np.all(np.diff(neuron_fit.loss_history[:40]) < 0)

### Watch optimization and fitting at the same time

The contour plot lives in parameter space: each point $(b,w)$ is a complete line. The line panels live in data space: they show what selected parameter states predict. The loss curve lives in iteration space. Learning becomes clearer when all three views are connected.

In [ ]:
bias_values = np.linspace(neuron_fit.bias - 160, neuron_fit.bias + 160, 150)
weight_values = np.linspace(neuron_fit.weights[0] - 120, neuron_fit.weights[0] + 120, 150)
B, W = np.meshgrid(bias_values, weight_values)
standardized_x = X_train[:, 0]
loss_surface = np.mean(
    (B[None, :, :] + W[None, :, :] * standardized_x[:, None, None] - y_train[:, None, None]) ** 2,
    axis=0,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].contour(B, W, loss_surface, levels=24)
axes[0].plot(
    neuron_fit.parameter_history[:, 0], neuron_fit.parameter_history[:, 1],
    color="crimson", marker="o", markersize=2,
)
axes[0].set(xlabel="bias $b$", ylabel="standardized-feature weight $w$", title="MSE surface and gradient-descent path")
axes[1].semilogy(neuron_fit.loss_history)
axes[1].set(xlabel="epoch", ylabel="training MSE (log scale)", title="Objective decreases toward its floor")
plt.tight_layout()
plt.show()

In [ ]:
raw_domain = np.linspace(X_train_raw.min(), X_train_raw.max(), 150).reshape(-1, 1)
scaled_domain = scaler.transform(raw_domain)
snapshot_epochs = (0, 1, 5, 160)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharex=True, sharey=True)
for ax, epoch in zip(axes, snapshot_epochs, strict=True):
    bias_at_epoch, weight_at_epoch = neuron_fit.parameter_history[epoch]
    prediction_at_epoch = scaled_domain[:, 0] * weight_at_epoch + bias_at_epoch
    ax.scatter(X_train_raw[:, 0], y_train, alpha=0.35, s=18)
    ax.plot(raw_domain[:, 0], prediction_at_epoch, color="crimson")
    ax.set_title(f"epoch {epoch}\nMSE={neuron_fit.loss_history[epoch]:.3f}")
    ax.set_xlabel("baseline BMI (kg/m²)")
axes[0].set_ylabel("one-year progression score")
plt.tight_layout()
plt.show()

### Compare three routes to the same OLS minimum

For an unregularized linear model, the normal equations, a stable least-squares solver, and converged gradient descent target the same global minimum. Their algorithms and numerical behavior differ, but the optimization problem is identical.

In [ ]:
reference_model = LinearRegression().fit(X_train, y_train)
augmented_design = np.column_stack([np.ones(len(X_train)), X_train])
least_squares_parameters = np.linalg.lstsq(augmented_design, y_train, rcond=None)[0]
neuron_test_prediction = X_test @ neuron_fit.weights + neuron_fit.bias
reference_test_prediction = reference_model.predict(X_test)

assert np.allclose(
    [neuron_fit.bias, *neuron_fit.weights], least_squares_parameters, atol=1e-8
)
assert np.allclose(neuron_test_prediction, reference_test_prediction, atol=1e-8)
print(f"held-out RMSE: {np.sqrt(np.mean((y_test - neuron_test_prediction) ** 2)):.2f} score points")

## 4. Why scaling changes gradient descent

With an augmented design matrix $\widetilde X=[\mathbf1\;X]$, the MSE Hessian is

$$H=\frac{2}{n}\widetilde X^T\widetilde X.$$

Its eigenvalues describe curvature. The condition number $\kappa(H)=\lambda_{\max}/\lambda_{\min}$ measures anisotropy when $H$ is positive definite. Large scale differences produce elongated contours: a learning rate safe in the steep direction makes progress slow in the shallow direction.

Scaling changes the path and useful learning rates. As shown in the previous notebook, it need not change the final unregularized OLS prediction class.

In [ ]:
raw_two_feature_design = np.column_stack([
    rng.normal(0, 1, 240),
    rng.normal(0, 900, 240),
])
scaled_two_feature_design = StandardScaler().fit_transform(raw_two_feature_design)
raw_augmented = np.column_stack([np.ones(len(raw_two_feature_design)), raw_two_feature_design])
scaled_augmented = np.column_stack([np.ones(len(scaled_two_feature_design)), scaled_two_feature_design])
raw_hessian = (2 / len(raw_augmented)) * raw_augmented.T @ raw_augmented
scaled_hessian = (2 / len(scaled_augmented)) * scaled_augmented.T @ scaled_augmented
raw_condition = np.linalg.cond(raw_hessian)
scaled_condition = np.linalg.cond(scaled_hessian)

assert scaled_condition < raw_condition
print(f"Hessian condition number: raw={raw_condition:,.0f}; scaled={scaled_condition:.2f}")

## 5. Batch, stochastic, mini-batch, and automatic differentiation

These terms describe different parts of training:

- **Batch gradient descent** computes one gradient from all training observations before each update. That is what we implemented.
- **Stochastic gradient descent (SGD)** updates from one sampled observation at a time. Its gradient is noisy but updates are cheap.
- **Mini-batch gradient descent** uses a subset. Vectorized hardware and controllable gradient noise make it standard for large models.
- **Automatic differentiation** computes derivatives of a program by the chain rule. It does not choose the loss, optimizer, data split, batch size, or learning rate.

An **epoch** means one complete pass through the training observations; the number of parameter updates per epoch depends on batch size. Shuffle only training rows, use seeded randomness when reproducibility matters, and never let validation or test rows enter an optimizer batch.

## 6. Where gradient methods appear

The same local mechanism supports very different models and scientific tasks:

| Objective | Parameters | Example application |
| --- | --- | --- |
| logistic negative log-likelihood | linear score coefficients | calibrated event prediction |
| matrix reconstruction loss | latent factors | imaging, compression, recommendation |
| simulation mismatch | physical or design parameters | inverse problems and system identification |
| energy or variational functional | fields or basis coefficients | mechanics, PDEs, scientific machine learning |
| cross-entropy over a composed network | millions or billions of weights | vision, language, molecular representation |

The optimizer only sees a scalar objective and derivatives. Domain meaning enters through the data, parameterization, constraints, and objective. Two projects can use Adam or SGD while making completely different claims.

## Common failure modes

### Learning rate too large

Loss oscillates upward, parameters explode, or nonfinite values appear. Stop early and preserve the failed trace; do not silently return the last parameters.

### Learning rate too small or poor conditioning

Loss decreases but useful progress is unreasonably slow. Inspect feature scales, Hessian/gradient behavior, and whether the stopping rule is meaningful.

### Incorrect gradient

A sign, factor, broadcasting, or reduction error can still produce changing loss. Compare analytic gradients with finite differences on a tiny deterministic fixture.

### Convergence confused with generalization

Reaching a low training loss only solves the empirical optimization problem. It says nothing by itself about leakage, distribution shift, or held-out error.

### Unstable stopping rule

“Stop when loss is small” has no universal scale. Track relative loss change, gradient norm, maximum epochs, and validation behavior with explicit tolerances.

## Debugging gradient descent

1. Check shapes: $X:(n,p)$, $\boldsymbol w:(p,)$, $y:(n,)$, predictions $(n,)$.
2. Evaluate the objective and gradient on a two-row hand calculation.
3. Run a central finite-difference gradient check.
4. Log step, training loss, validation loss, gradient norm, parameter norm, and learning rate.
5. Assert finiteness and fail loudly at the first NaN or infinity.
6. Plot loss on both linear and logarithmic axes.
7. Standardize from training statistics and record the transformer.
8. Compare with `np.linalg.lstsq` or `LinearRegression` on the same data.

## Professional practice

A training implementation should make its contract testable:

- typed array shapes and validated finite inputs;
- NumPy-style docstrings stating units and return history;
- deterministic fixtures for gradient and convergence tests;
- explicit maximum iterations and nonfinite failure behavior;
- structured telemetry rather than print-only progress;
- separate training, validation, and final evaluation entry points; and
- a versioned configuration containing optimizer, learning rate, batch size, seed, stopping rule, and feature transformer.

A notebook is ideal for seeing the path. Repeatable training belongs in tested modules and scripts so CI can verify it and production can reproduce it.

## Guided practice: predict before you run

For $g(\theta)=4(\theta+2)^2$, derive $g'(\theta)$ and the exact stable interval for a fixed learning rate. Then implement the derivative and run from $\theta_0=3$ using one stable and one unstable rate.

**Success criteria:** show the error recurrence, state the interval before executing code, and plot both objective histories.

In [ ]:
def steeper_gradient(parameter: float | np.ndarray) -> float | np.ndarray:
    """Return the derivative of ``4 * (parameter + 2)**2``."""

    return 8.0 * (parameter + 2.0)


def run_with_gradient(start: float, rate: float, steps: int) -> np.ndarray:
    history = [float(start)]
    for _ in range(steps):
        history.append(history[-1] - rate * steeper_gradient(history[-1]))
    return np.asarray(history)


stable_trace = run_with_gradient(3.0, 0.10, 15)
unstable_trace = run_with_gradient(3.0, 0.30, 15)
assert abs(stable_trace[-1] + 2.0) < abs(stable_trace[0] + 2.0)
assert abs(unstable_trace[-1] + 2.0) > abs(unstable_trace[0] + 2.0)

## Independent practice

Remove standardization from the linear-neuron example. Search for a stable learning rate without looking at test labels. Compare the loss path, useful step-size range, and final least-squares prediction with the standardized version.

**Success criteria:** use training data only to choose the rate, preserve a maximum epoch limit, and distinguish optimization speed from held-out predictive quality.

## Extension: implement mini-batches with momentum

Add seeded shuffling, configurable batch size, and velocity

$$
\boldsymbol v_{t+1}=\gamma\boldsymbol v_t+\nabla L_{B_t}(\boldsymbol\theta_t),
\qquad
\boldsymbol\theta_{t+1}=\boldsymbol\theta_t-\eta\boldsymbol v_{t+1}.
$$

**Success criteria:** define what an epoch means, cover every training row once per epoch, test reproducibility, compare against the OLS solution, and log both batch and epoch-level loss without touching test targets.

## Retrieval practice

1. What local approximation does a derivative provide?
2. Why is $-\nabla J$ steepest descent under the Euclidean norm?
3. Derive the stable learning-rate interval for the scalar quadratic.
4. Derive $\partial L/\partial w_j$ for the identity neuron.
5. Which choices turn a neuron into linear regression?
6. Why can scaling change the optimization path but not the final OLS prediction class?
7. How do batch gradient descent, SGD, mini-batches, and automatic differentiation differ?
8. Why does convergence not establish generalization?

## Takeaway

A derivative describes local sensitivity; its negative supplies a downhill direction; the learning rate chooses how far to trust that local information. The gradient extends the idea to many parameters. For an identity-activation neuron under mean squared error, the resulting objective is ordinary least squares, and gradient descent approaches the same global minimum as a stable linear-algebra solver.

The optimization path is only one layer of an ML system. The prediction contract, data boundary, evaluation, and operational monitoring remain just as important.

## Further reading

- [NumPy: gradients with `numpy.gradient`](https://numpy.org/doc/stable/reference/generated/numpy.gradient.html)
- [NumPy: least-squares solver](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)
- [scikit-learn: ordinary least squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- [scikit-learn: diabetes dataset description and provenance](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset)
- [scikit-learn: scaling and preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Python tutorial: defining functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions)